# 3) Retrieval Evaluation & Query Correction
## Corrective RAG System — Step 3 of 4

This is the heart of **Corrective RAG**:

1. Retrieve the top-k chunks for the question.
2. Grade each chunk with an LLM: relevant or not relevant to the question (relevance grading).
3. If the fraction of relevant chunks is below a threshold, rewrite the question with an LLM and
   retrieve again.
4. Repeat for a limited number of attempts (`max_rewrites`), then return the best result available.

This notebook covers:
- Retrieve and evaluate relevant chunks
- Rewrite the query and re-retrieve if context is weak
- Detect irrelevant or low-quality retrieved context


In [1]:
import os
import urllib.request
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma

load_dotenv()

PROJECT_ROOT = Path.cwd()
VECTORSTORE_DIR = PROJECT_ROOT / "vectorstore" / "chroma_db"
COLLECTION_NAME = "crag_course_docs"
EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2:3b"
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

try:
    urllib.request.urlopen(OLLAMA_BASE_URL, timeout=3)
except Exception as exc:
    raise RuntimeError(
        f"Could not reach Ollama at {OLLAMA_BASE_URL}. Make sure the Ollama app is running "
        "(it starts automatically after installation, or run 'ollama serve' manually)."
    ) from exc

if not VECTORSTORE_DIR.exists():
    raise FileNotFoundError("No vector store found - run notebook 02_Embeddings_VectorStore first.")

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(VECTORSTORE_DIR),
)
llm = ChatOllama(model=LLM_MODEL, temperature=0, base_url=OLLAMA_BASE_URL)

print(f"Vector store loaded with {vectorstore._collection.count()} vectors")


Vector store loaded with 11 vectors


## Step 1 — Retriever

In [2]:
def retrieve(query: str, k: int = 4):
    return vectorstore.similarity_search(query, k=k)


## Step 2 — Relevance grader (LLM-as-judge)

We use `with_structured_output` to guarantee the model returns only `yes`/`no`, not free text.

In [3]:
class GradeDocument(BaseModel):
    """Binary relevance grade for a retrieved document."""

    binary_score: Literal["yes", "no"] = Field(
        description="'yes' if the document is relevant to the question, otherwise 'no'"
    )


GRADER_SYSTEM_PROMPT = (
    "You are a grader assessing the relevance of a retrieved document to a user question.\n"
    "If the document contains information that helps answer the question, grade it as relevant.\n"
    "This does not need to be a strict, exact match — the goal is to filter out clearly irrelevant "
    "or off-topic retrievals, not to be overly strict."
)

grader_llm = llm.with_structured_output(GradeDocument)


def grade_document(question: str, document_text: str) -> str:
    prompt = f"Retrieved document:\n\n{document_text}\n\nUser question: {question}"
    result = grader_llm.invoke(
        [
            {"role": "system", "content": GRADER_SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
    )
    return result.binary_score


## Step 3 — Query rewriter

In [4]:
REWRITER_SYSTEM_PROMPT = (
    "You are a query re-writer that converts an input question into a better version optimized for "
    "vector store retrieval. Look at the input and try to reason about the underlying semantic intent, "
    "and make the question more specific and information-dense. Return only the rewritten question."
)


def rewrite_query(question: str) -> str:
    response = llm.invoke(
        [
            {"role": "system", "content": REWRITER_SYSTEM_PROMPT},
            {"role": "user", "content": f"Original question: {question}\n\nRewritten question:"},
        ]
    )
    return response.content.strip()


## Step 4 — The corrective retrieval loop

In [5]:
def corrective_retrieve(
    question: str,
    k: int = 4,
    relevance_threshold: float = 0.5,
    max_rewrites: int = 2,
):
    """Retrieve -> grade -> (rewrite -> retrieve again) until enough relevant context is found."""
    trace = []
    current_query = question
    best_relevant_docs: list = []

    for attempt in range(max_rewrites + 1):
        docs = retrieve(current_query, k=k)
        grades = [grade_document(question, doc.page_content) for doc in docs]
        relevant_docs = [doc for doc, grade in zip(docs, grades) if grade == "yes"]
        ratio = len(relevant_docs) / len(docs) if docs else 0.0

        trace.append(
            {
                "attempt": attempt,
                "query_used": current_query,
                "retrieved": len(docs),
                "relevant": len(relevant_docs),
                "relevance_ratio": round(ratio, 2),
            }
        )

        if len(relevant_docs) > len(best_relevant_docs):
            best_relevant_docs = relevant_docs

        if ratio >= relevance_threshold:
            return relevant_docs, trace

        if attempt < max_rewrites:
            current_query = rewrite_query(current_query)

    return best_relevant_docs, trace


## Step 5 — Demo: happy path (a question that matches the documents well)

In [6]:
question_ok = "How does Corrective RAG reduce hallucination compared to standard RAG?"
relevant_docs, trace = corrective_retrieve(question_ok)

for step in trace:
    print(step)

print(f"\nFinal relevant chunks used: {len(relevant_docs)}")
for doc in relevant_docs:
    print(f" - {doc.metadata['source']} (chunk {doc.metadata['chunk_index']})")


{'attempt': 0, 'query_used': 'How does Corrective RAG reduce hallucination compared to standard RAG?', 'retrieved': 4, 'relevant': 4, 'relevance_ratio': 1.0}

Final relevant chunks used: 4
 - 01_rag_basics.txt (chunk 2)
 - 03_corrective_rag.txt (chunk 1)
 - 03_corrective_rag.txt (chunk 3)
 - 03_corrective_rag.txt (chunk 0)


## Step 6 — Demo: correction path (a vague / out-of-scope question)

In [7]:
question_vague = "Tell me about the weather forecasting models used in 1990s aviation."
relevant_docs_2, trace_2 = corrective_retrieve(question_vague)

for step in trace_2:
    print(step)

print(f"\nFinal relevant chunks used: {len(relevant_docs_2)}")
if not relevant_docs_2:
    print("No relevant context found even after query rewriting — this question is out of scope for our documents.")


{'attempt': 0, 'query_used': 'Tell me about the weather forecasting models used in 1990s aviation.', 'retrieved': 4, 'relevant': 1, 'relevance_ratio': 0.25}
{'attempt': 1, 'query_used': 'What were the primary numerical weather prediction (NWP) models employed by the National Weather Service (NWS) and the Federal Aviation Administration (FAA) during the 1990s, specifically for short-term forecast accuracy and their impact on flight planning and air traffic control?', 'retrieved': 4, 'relevant': 1, 'relevance_ratio': 0.25}
{'attempt': 2, 'query_used': 'What were the specific numerical weather prediction (NWP) models used by the National Weather Service (NWS) and the Federal Aviation Administration (FAA) in the early 1990s, such as the Global Forecast System (GFS) and the North American Mesoscale Forecast System (NAM), for predicting short-term weather patterns (<24 hours) and their direct impact on flight planning and air traffic control operations during that period?', 'retrieved': 4, '

> **Note:** try changing `k`, `relevance_threshold`, `max_rewrites` in `corrective_retrieve` and see how they affect the number of rewrite attempts.